In [ ]:
import os
import pandas as pd
import numpy as np
import time
import json
from tqdm import tqdm
import concurrent.futures
import google.generativeai as genai
from google.generativeai.types import content_types

# ==========================================
# 1. API Configuration & Schema Definition
# ==========================================
API_KEY = os.environ.get("GEMINI_API_KEY")
genai.configure(api_key=API_KEY)

# Define the schema using a standard Python dictionary to prevent AttributeError
assay_info_schema = {
    "type": "OBJECT",
    "properties": {
        "assay_type": {
            "type": "STRING",
            "description": "Choose exactly one: 'cAMP', 'Ca2+', 'IP1', 'GTPgS', 'beta-arrestin', 'binding_only', or 'reporter_other'."
        },
        "target_moa": {
            "type": "STRING",
            "description": "Choose exactly one: 'Agonist', 'Antagonist', 'Partial Agonist', 'Inverse Agonist', 'PAM', 'NAM', or 'Unknown'."
        },
        "is_negation": {
            "type": "BOOLEAN",
            "description": "True ONLY if the assay explicitly failed to show the target MoA, otherwise False."
        },
        "disease_context": {
            "type": "STRING",
            "description": "Target disease, clinical indication, or therapeutic area. 'N/A' if not stated."
        },
        "cell_line": {
            "type": "STRING",
            "description": "Specific cell line used (e.g., HEK293, CHO, HeLa). 'N/A' if not stated."
        }
    },
    "required": ["assay_type", "target_moa", "is_negation", "disease_context", "cell_line"]
}

model = genai.GenerativeModel(
    'models/gemini-2.5-flash',
    generation_config={
        "response_mime_type": "application/json",
        "response_schema": assay_info_schema,
        "temperature": 0.1 # Low temperature for factual extraction
    }
)

In [ ]:
# ==========================================
# 2. LLM Extraction Functions
# ==========================================
def extract_assay_info_llm(text, retries=3):
    """Calls the Gemini API with schema enforcement and retry logic."""
    if pd.isna(text) or str(text).strip() == "":
        return {"assay_type": "Unknown", "target_moa": "Unknown", "is_negation": False, "disease_context": "N/A", "cell_line": "N/A"}

    prompt = f"Analyze the following GPCR assay description and extract the required metadata.\n\nAssay Description: \"{text}\""
    
    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return json.loads(response.text)
        except Exception as e:
            if "429" in str(e) or "ResourceExhausted" in str(e):
                time.sleep(5)
            else:
                print(f"Extraction error on attempt {attempt+1}: {e}")
                time.sleep(2)
                
    return {"assay_type": "Unknown", "target_moa": "Unknown", "is_negation": False, "disease_context": "N/A", "cell_line": "N/A"}

def process_unique_descriptions(df, text_column):
    """Extracts unique descriptions, processes them via LLM, and maps back to the dataframe."""
    unique_texts = df[text_column].dropna().unique().tolist()
    print(f"Mining {len(unique_texts)} unique assay descriptions out of {len(df)} total rows...")
    
    extracted_data = {}
    
    def process_single_text(text):
        return text, extract_assay_info_llm(text)

    # Use ThreadPoolExecutor for parallel API requests
    with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
        futures = {executor.submit(process_single_text, text): text for text in unique_texts}
        for future in tqdm(concurrent.futures.as_completed(futures), total=len(unique_texts), desc="LLM Processing"):
            text, result = future.result()
            extracted_data[text] = result

    # Map results back
    df['Assay_Type_LLM'] = df[text_column].map(lambda x: extracted_data.get(x, {}).get('assay_type', 'Unknown'))
    df['Target_MoA_LLM'] = df[text_column].map(lambda x: extracted_data.get(x, {}).get('target_moa', 'Unknown'))
    df['Is_Negation'] = df[text_column].map(lambda x: extracted_data.get(x, {}).get('is_negation', False))
    df['Disease_Context'] = df[text_column].map(lambda x: extracted_data.get(x, {}).get('disease_context', 'N/A'))
    df['Cell_Line'] = df[text_column].map(lambda x: extracted_data.get(x, {}).get('cell_line', 'N/A'))
    
    return df

In [ ]:
# ==========================================
# 3. Data Normalization & MoA Labeling
# ==========================================
def normalize_value_um(val, unit):
    """Normalizes affinity/activity values to micromolar (uM)."""
    try:
        val = float(val)
        unit = str(unit).lower().strip()
        if 'nm' in unit: return val / 1000.0
        elif 'um' in unit: return val
        elif 'mm' in unit: return val * 1000.0
        else: return val / 1000.0 # Default fallback
    except:
        return np.nan

def assign_main_moa(row):
    """Assigns final MoA based on biological hierarchy (Layer 1-4) and 10uM threshold."""
    val = row['Value_uM']
    assay = row['Assay_Type_LLM']
    moa = row['Target_MoA_LLM']
    negation = row['Is_Negation']
    
    if pd.isna(val):
        return 'Unknown'
        
    # 1. Inactive / Non-binder (Threshold > 10uM or Explicit Negation)
    if val > 10.0 or negation == True:
        if assay == 'binding_only':
            return 'Non-binder'
        else:
            return 'Inactive'
            
    # 2. Active (Threshold <= 10uM)
    if assay in ['cAMP', 'Ca2+', 'IP1', 'GTPgS']: # Layer 2: Proximal G-protein
        if moa == 'Agonist': return 'Full Agonist'
        if moa == 'Antagonist': return 'Antagonist'
        return moa 
    elif assay == 'beta-arrestin':                # Layer 3: Biased Pathway
        return 'Beta-arrestin-biased'
    elif assay == 'reporter_other':               # Layer 4: Distal/Reporter
        return 'Reporter-based'
    elif assay == 'binding_only':                 # Layer 1: Binding Only
        return 'Binder_Unknown_MoA'
        
    return 'Unknown'

In [ ]:
# ==========================================
# 4. Main Execution Pipeline
# ==========================================
if __name__ == "__main__":
    
    # --- A. Load Human GPCR Target List ---
    human_gpcr_df = pd.read_csv('./Input/Human_GPCR_PDB_Info.csv')
    human_uniprot_list = human_gpcr_df['Entry'].dropna().unique().tolist()
    print(f"Loaded {len(human_uniprot_list)} Human GPCR UniProt IDs.")

    # --- B. Process ChEMBL ---
    print("\n[Processing ChEMBL]")
    chembl_assay = pd.read_csv('./Output/DB/ChEMBL/NAR/ChEMBL_v36_Assays_Single_Direct.csv')
    chembl_target = pd.read_csv('./Output/DB/ChEMBL/NAR/ChEMBL_v36_Target_Metadata.csv')
    
    # Merge to get UniProt Accession and filter for Human GPCRs BEFORE LLM processing
    chembl_merged = pd.merge(chembl_assay, chembl_target[['tid', 'uniprot_accession']], on='tid', how='left')
    chembl_filtered = chembl_merged[chembl_merged['uniprot_accession'].isin(human_uniprot_list)].copy()
    
    if not chembl_filtered.empty:
        chembl_filtered = process_unique_descriptions(chembl_filtered, 'assay_description')
        chembl_filtered['Ligand_InChIKey'] = chembl_filtered['standard_inchi_key']
        chembl_filtered['GPCR_UniProt'] = chembl_filtered['uniprot_accession']
        chembl_filtered['Value_uM'] = chembl_filtered.apply(lambda x: normalize_value_um(x['standard_value'], x['standard_units']), axis=1)
        chembl_filtered['Source_DB'] = 'ChEMBL'
    else:
        print("No matching Human GPCR targets found in ChEMBL dataset.")

    # --- C. Process BindingDB ---
    print("\n[Processing BindingDB]")
    bdb_df = pd.read_csv('./Output/DB/BindingDB/NAR/BindingDB_gpcr_assay_desc_NAR.csv')
    
    # Filter for Human GPCRs BEFORE LLM processing
    bdb_filtered = bdb_df[bdb_df['Entry'].isin(human_uniprot_list)].copy()
    
    if not bdb_filtered.empty:
        bdb_filtered = process_unique_descriptions(bdb_filtered, 'DESCRIPTION')
        bdb_filtered['Ligand_InChIKey'] = bdb_filtered['Ligand InChI Key']
        bdb_filtered['GPCR_UniProt'] = bdb_filtered['Entry']
        bdb_filtered['Value_uM'] = bdb_filtered.apply(lambda x: normalize_value_um(x['Affinity'], x['Type']), axis=1)
        bdb_filtered['Source_DB'] = 'BindingDB'
    else:
        print("No matching Human GPCR targets found in BindingDB dataset.")

    # --- D. Merge and Finalize Labels ---
    print("\n[Merging Datasets and Assigning Final Hierarchy Labels]")
    columns_to_keep = [
        'Ligand_InChIKey', 'GPCR_UniProt', 'Source_DB', 'Assay_Type_LLM', 
        'Target_MoA_LLM', 'Is_Negation', 'Value_uM', 'Disease_Context', 'Cell_Line'
    ]
    
    datasets_to_concat = []
    if not chembl_filtered.empty: datasets_to_concat.append(chembl_filtered[columns_to_keep])
    if not bdb_filtered.empty: datasets_to_concat.append(bdb_filtered[columns_to_keep])
    
    if datasets_to_concat:
        combined_df = pd.concat(datasets_to_concat, ignore_index=True)
        combined_df.dropna(subset=['Ligand_InChIKey', 'GPCR_UniProt'], inplace=True)
        
        # Apply the Layer 1-4 logic
        combined_df['Final_MoA'] = combined_df.apply(assign_main_moa, axis=1)
        
        # Save Final Output
        os.makedirs('./Output/DB/GPCRactDB/', exist_ok=True)
        output_path = './Output/DB/GPCRactDB/ChEMBL_BDB_Assay_Labeled.csv'
        combined_df.to_csv(output_path, index=False)
        print(f"\nSuccessfully saved final dataset to: {output_path}")
        print(combined_df['Final_MoA'].value_counts())
    else:
        print("Pipeline aborted: No data available to merge.")

In [ ]:
df = pd.read_csv('./Output/DB/GPCRactDB/ChEMBL_BDB_Assay_Labeled.csv')
# 허용된 MoA 리스트가 아니면 Unknown으로 강제 변환
allowed_moa = ['Agonist', 'Antagonist', 'Partial Agonist', 'Inverse Agonist', 'PAM', 'NAM', 'Unknown', 'binding_only']
df.loc[~df['Target_MoA_LLM'].isin(allowed_moa), 'Final_MoA'] = 'Unknown'

allowed_moa = ['Full Agonist', 'Antagonist', 'Partial Agonist', 'Inverse Agonist', 'PAM', 'NAM', 'Beta-arrestin-biased', 'Reporter-based', 'Binder_Unknown_MoA', 'Inactive', 'Non-binder']
df.loc[~df['Final_MoA'].isin(allowed_moa), 'Final_MoA'] = 'Unknown'

In [ ]:
df.to_csv('./Output/DB/GPCRactDB/ChEMBL_BDB_Assay_Labeled.csv', index = None)

In [ ]:
df

In [ ]:
tb['Assay_Type_LLM'].value_counts()

In [ ]:
tb['Cell_Line'].value_counts()

In [ ]:
tb